# Single-modality baseline models: Random Forest

### Preparation

In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, KFold, GroupShuffleSplit

In [ ]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl")
df.head()

In [29]:
def genomic_baseline_model(df, target, multi=True, single=False):
    if multi:
        print(f"\n--- Genomic Baseline for Target: {target} ---")
        # training set for genomic features
        X_genomic = df.filter(regex=r'.* \(.*\)').values # L1000 landmark genes
        y_genomic = df[target].values # y: drug response
        # train-test split
        gss = GroupShuffleSplit(test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(X_genomic, y_genomic, groups=df['ModelID'].values))
        X_train, X_test = X_genomic[train_idx], X_genomic[test_idx]
        y_train, y_test = y_genomic[train_idx], y_genomic[test_idx]
        groups = df.iloc[train_idx]['ModelID'].values # to ensure held-out validation

        # initialize the Random Forest Regressor
        rf_genomic = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42
        )

        # perform Cross-Validation
        print("Starting Cross-Validation on Training Data...")
        cv_results = cross_validate(
            rf_genomic, X_train, y_train, 
            groups=groups, 
            cv=GroupKFold(n_splits=5),
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        # output Results
        mse_scores = -cv_results['test_neg_mean_squared_error']
        rmse_scores = np.sqrt(mse_scores)
        r2_scores = cv_results['test_r2']

        print(f"--- Genomic Baseline Performance ---")
        print(f"R² Score: {np.mean(r2_scores):.4f}")
        print(f"RMSE:     {np.mean(rmse_scores):.4f}")
        print(f"------------------------------------")

        # test fit
        rf_genomic.fit(X_train, y_train)
        y_pred = rf_genomic.predict(X_test)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Test R² Score: {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        
        importances = pd.Series(rf_genomic.feature_importances_, index=df.filter(regex=r'.* \(.*\)').columns.values)
        print("\nTop 5 Genetic Features:")
        print(importances.sort_values(ascending=False).head(5))
    
    # training set for single DRUG analysis
    if single:
        print(f"\nSingle drug testing with DrugID: {df['DRUG_ID'].value_counts().index[0]} for target: {target}")
        df_single_drug = df[df['DRUG_ID'] == df['DRUG_ID'].value_counts().index[0]]
        X_single_d = df_single_drug.filter(regex=r'.* \(.*\)').values
        y_single_d = df_single_drug.dropna(subset=[target])[target].values

        # initialize a new Random Forest for this single drug
        rf_single = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            n_jobs=-1,
            random_state=42
        )

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_results_single = cross_validate(
            rf_single, X_single_d, y_single_d, 
            cv=kf,
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )
        print(f"------------------------------------")
        print(f"R² Score: {np.mean(cv_results_single['test_r2']):.4f}")
        print(f"RMSE:     {np.mean(np.sqrt(-cv_results_single['test_neg_mean_squared_error'])):.4f}")
        print(f"Standard deviation of target data: {np.std(y_single_d):.4f}")
        print(f"------------------------------------")
        rf_single.fit(X_single_d, y_single_d)
        importances_single = pd.Series(rf_single.feature_importances_, index=df_single_drug.filter(regex=r'.* \(.*\)').columns.values)
        print("Top 5 Genetic Features for Single Drug:")
        print(importances_single.sort_values(ascending=False).head(5))

In [4]:
def chemical_aggregated_baseline_model(df, target, multi=True, single=False):
    if "Bit_0" not in df.columns:
        fp_df = pd.DataFrame(df['MorganFP'].tolist())
        fp_df.columns = [f'Bit_{i}' for i in fp_df.columns]
        df = pd.concat([df, fp_df], axis=1)
    if multi:
        # training set for chemical features
        # aggregate data: calculate mean target value per drug
        X_cols = df.loc[:, df.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].columns.tolist()
        df_drug_baseline = df.groupby('DRUG_ID').agg({
            target: 'mean',
            **{col: 'first' for col in X_cols}
        }).reset_index()

        # define Features and Target
        X_chem_bl = df_drug_baseline[X_cols].values.astype(float)
        y_chem_bl = df_drug_baseline[target].values

        rf_model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            random_state=42,
            max_features='sqrt',
            n_jobs=-1
        )

        # cross-validation setup
        print("Starting Cross-Validation on Aggregated Chemical Data...")
        cv_results = cross_validate(
            rf_model, X_chem_bl, y_chem_bl, 
            cv=GroupKFold(n_splits=5),
            groups=df_drug_baseline['DRUG_ID'].values,
            scoring=['r2', 'neg_mean_squared_error'],
            return_train_score=True
        )

        print(f"--- Chemical features baseline performance ---")
        print(f"R² Score: {np.mean(cv_results['test_r2']):.4f}")
        print(f"RMSE:     {np.mean(np.sqrt(-cv_results['test_neg_mean_squared_error'])):.4f}")
        print(f"------------------------------------")

        # feature Importance
        rf_model.fit(X_chem_bl, y_chem_bl)
        importances = pd.Series(rf_model.feature_importances_, index=X_cols)
        print("\nTop 5 chemical drivers for general drug potency:")
        print(importances.sort_values(ascending=False).head(5))

    if single:
        # training set for single CELL LINE analysis
        print(f"Single cell line testing with ModelID: {df['ModelID'].value_counts().index[0]} for target: {target}")
        df_single_cell = df[df['ModelID'] == df['ModelID'].value_counts().index[0]]
        X_single_c = df_single_cell.loc[:, df.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].values.astype(float)
        y_single_c = df_single_cell[target].values

        rf_single_c = RandomForestRegressor(
            n_estimators=100, 
            max_depth=5,
            min_samples_leaf=10, 
            n_jobs=-1, 
            random_state=42
        )

        print(f"Target Variance: {np.var(y_single_c):.4f}")
        # grouping after drugs to make sure that one drug is not split between train and test sets
        cv_results_single = cross_validate(
            rf_single_c, X_single_c, y_single_c, 
            cv=GroupKFold(n_splits=5),
            groups=df_single_cell['DRUG_ID'].values,
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        print(f"------------------------------------")
        print(f"Single cell line R² Score: {np.mean(cv_results_single['test_r2']):.4f}")
        print(f"RMSE:     {np.mean(np.sqrt(-cv_results_single['test_neg_mean_squared_error'])):.4f}")
        print(f"Standard deviation of target data: {np.std(y_single_c):.4f}")
        print(f"------------------------------------")

        rf_single_c.fit(X_single_c, y_single_c)
        importances_single = pd.Series(rf_single_c.feature_importances_,
                                       index= df_single_cell.loc[:, df.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].columns)
        print("\nTop 5 drug features for chosen single cell line:")
        print(importances_single.sort_values(ascending=False).head(5))

### Single testing

In [23]:
genomic_baseline_model(df, target='LN_IC50', multi=False, single=True)
genomic_baseline_model(df, target='AUC', multi=False, single=True)


Single drug testing with DrugID: 1862 for target: LN_IC50
------------------------------------
R² Score: 0.2018
RMSE:     0.6284
Standard deviation of target data: 0.7073
------------------------------------
Top 5 Genetic Features for Single Drug:
TSKU (25987)      0.044599
IKZF1 (10320)     0.025462
FBXL12 (54850)    0.021967
ERBB3 (2065)      0.021069
UGDH (7358)       0.019308
dtype: float64

Single drug testing with DrugID: 1862 for target: AUC
------------------------------------
R² Score: 0.2023
RMSE:     0.0820
Standard deviation of target data: 0.0922
------------------------------------
Top 5 Genetic Features for Single Drug:
TSKU (25987)      0.024127
APP (351)         0.022788
FBXL12 (54850)    0.022685
IKZF1 (10320)     0.020567
PTPRF (5792)      0.016836
dtype: float64


In [34]:
chemical_aggregated_baseline_model(df, target='AUC', multi=False, single=True)
chemical_aggregated_baseline_model(df, target='LN_IC50', multi=False, single=True)

Single cell line testing with ModelID: ACH-000672 for target: AUC
Target Variance: 0.0056
------------------------------------
Single cell line R² Score: -0.1873
RMSE:     0.0700
Standard deviation of target data: 0.0748
------------------------------------

Top 5 drug features for chosen single cell line:
Bit_47              0.207426
LumpedHydrophobe    0.206672
Bit_380             0.097492
Bit_233             0.059454
Bit_101             0.043178
dtype: float64
Single cell line testing with ModelID: ACH-000672 for target: LN_IC50
Target Variance: 4.5754
------------------------------------
Single cell line R² Score: 0.0598
RMSE:     2.0442
Standard deviation of target data: 2.1390
------------------------------------

Top 5 drug features for chosen single cell line:
Hydrophobe    0.112152
Bit_380       0.085764
Bit_11        0.080067
Bit_541       0.079577
Acceptor      0.075844
dtype: float64


### Defining training sets

In [ ]:
# training set for genomic features
X_genomic = df.filter(regex=r'.* \(.*\)').values # L1000 landmark genes (genomic features)
y_genomic = df['AUC'].values # y: drug response

# training set for single DRUG analysis
print(f"\nSingle drug testing with DrugID: {df['DRUG_ID'].value_counts().index[0]}")
df_single_drug = df[df['DRUG_ID'] == df['DRUG_ID'].value_counts().index[0]]
X_single_d = df_single_drug.filter(regex=r'.* \(.*\)').values
y_single_d = df_single_drug['AUC'].values

#####################################################################################
# training set for chemical features
X_chem = pd.concat([df.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']], df.loc[:, df.columns.str.startswith('Bit_')]], axis=1)
y_chem = df['AUC'].values

# training set for single CELL LINE analysis
print(f"Single cell line testing with ModelID: {df['ModelID'].value_counts().index[0]}")
df_single_cell = df[df['ModelID'] == df['ModelID'].value_counts().index[0]]
X_single_c = df_single_cell.loc[:, df.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].values.astype(float)
y_single_c = df_single_cell['AUC'].values

## Genomic features only

### Random Forest

In [30]:
genomic_baseline_model(df, "LN_IC50", multi=True, single=False)


--- Genomic Baseline for Target: LN_IC50 ---
Starting Cross-Validation on Training Data...
--- Genomic Baseline Performance ---
R² Score: 0.0491
RMSE:     2.7559
------------------------------------
Test R² Score: 0.0401
Test RMSE:     2.7539

Top 5 Genetic Features:
TJP1 (7082)       0.348390
SQSTM1 (8878)     0.060552
TBPL1 (9519)      0.027700
MACF1 (23499)     0.023398
APPBP2 (10513)    0.014722
dtype: float64


In [31]:
genomic_baseline_model(df, "AUC", multi=True, single=False)


--- Genomic Baseline for Target: AUC ---
Starting Cross-Validation on Training Data...
--- Genomic Baseline Performance ---
R² Score: 0.0227
RMSE:     0.1479
------------------------------------
Test R² Score: 0.0183
Test RMSE:     0.1435

Top 5 Genetic Features:
TJP1 (7082)       0.343662
SQSTM1 (8878)     0.023593
NOLC1 (9221)      0.008544
KLHDC2 (23588)    0.008047
TSPAN3 (10099)    0.007136
dtype: float64


## Chemical features only

In [ ]:
# get pharmacophore features as separate columns & concatenate back to the main dataframe
expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist())
df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
df = df.fillna(0)
#print(df.head())

### Random forest

Random forest with complete data (no aggregation but GroupKFold)

In [ ]:
# this i didn't test yet, it's to find the best parameters for chemical only-Random forest model
# can test which parameters are best for the chemical baseline model
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

# 1. Das Basis-Modell kriegt KEINE Liste!
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

# 2. Die Liste gehört NUR hier hinein:
param_distributions = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 15, 20, None],
    'max_features': ['sqrt', 'log2', 1.0, 0.33] # <-- Hier ist es absolut richtig!
}

# 3. Die Suche füttert dem Modell dann nacheinander jeweils einen Wert aus der Liste
rf_random = RandomizedSearchCV(
    estimator=rf_base,                  # Dein Basis-Modell
    param_distributions=param_distributions, # Dein Such-Raster
    n_iter=20,                          # Wie viele zufällige Kombis getestet werden sollen
    cv=5, 
    scoring='r2', 
    random_state=42, 
    n_jobs=-1
)

# Jetzt kannst du die Suche auf deinen Daten laufen lassen:
# rf_random.fit(X_chem_bl, y_chem_bl)

In [ ]:
def randomForest_chemical(df, target):
    # remove duplicate rows
    # not sure if this is the right approach, but it can help to reduce noise in the data and speed up training
    df.drop_duplicates(subset=['SMILES', target], inplace=True)
    # get MFP in separate columns
    if 'Bit_0' not in df.columns:
        # get MorganFP as separate columns
        fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index)
        fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
        df = pd.concat([df, fp_df], axis=1)
    # get pharmacophore features as separate columns
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist())
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)
    # training set for chemical features
    X_chem = pd.concat([df.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']],
                        df.loc[:, df.columns.str.startswith('Bit_')]], axis=1)
    X_cols = X_chem.columns.tolist()
    X = X_chem.values.astype(float)
    y = df[target].values
    # Train-Test-Split (80% Training/Validierung, 20% Test)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df['DRUG_ID']))
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Check, ob der Split geklappt hat:
    #drugs_train = set(df.iloc[train_idx]['DRUG_ID'])
    #drugs_test = set(df.iloc[test_idx]['DRUG_ID'])

    #overlap = drugs_train.intersection(drugs_test)
    #print(f"Anzahl überlappender Drogen: {len(overlap)}") # MUSS 0 sein!
    #print(f"Anzahl Drogen im Training: {len(drugs_train)}")
    #print(f"Anzahl Drogen im Testset: {len(drugs_test)}")

    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        max_features='sqrt'
        )

    # Use of GroupKFold makes sure that no data leakage is happening in CV
    print("Starting Cross-Validation on Training Data...")
    cv_results = cross_validate(
        rf_model, X_train, y_train, 
        cv=GroupKFold(n_splits=5),
        groups=df.iloc[train_idx]['DRUG_ID'].values,
        scoring=['r2', 'neg_mean_squared_error'],
        return_train_score=False
    )

    mean_cv_r2 = np.mean(cv_results['test_r2'])
    mean_cv_mse = -np.mean(cv_results['test_neg_mean_squared_error'])
    mean_cv_rmse = np.sqrt(mean_cv_mse)

    print(f"\n--- Cross-Validation Results (Train Data) ---")
    print(f"R² Score: {mean_cv_r2:.4f}")
    print(f"RMSE: {mean_cv_rmse:.4f}")

    # Build a forest of trees from the training set (X, y)
    rf_model.fit(X_train, y_train)

    # evaluation on unknown data set
    y_pred = rf_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)
    test_mse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n--- Test Set Results ---")
    print(f"Test R² Score: {test_r2:.4f}")
    print(f"Test RMSE: {test_mse:.4f}")

    # 7. Feature Importance ausgeben
    importances = pd.Series(rf_model.feature_importances_, index=X_cols)
    print("\nTop 5 Chemical Drivers for Drug Potency:")
    print(importances.sort_values(ascending=False).head(5))

In [15]:
randomForest_chemical(df, 'AUC')
randomForest_chemical(df, 'LN_IC50')

Anzahl überlappender Drogen: 0
Anzahl Drogen im Training: 191
Anzahl Drogen im Testset: 48
Starting Cross-Validation on Training Data...

--- Cross-Validation Results (Train Data) ---
R² Score: -0.0915
RMSE: 0.1726

--- Test Set Results ---
Test R² Score: -0.0853
Test RMSE: 0.1214

Top 5 Chemical Drivers for Drug Potency:
Bit_565    0.016398
Bit_693    0.014606
Bit_944    0.014550
Bit_569    0.013609
Bit_904    0.013394
dtype: float64
Anzahl überlappender Drogen: 0
Anzahl Drogen im Training: 191
Anzahl Drogen im Testset: 48
Starting Cross-Validation on Training Data...

--- Cross-Validation Results (Train Data) ---
R² Score: -0.0670
RMSE: 2.9366

--- Test Set Results ---
Test R² Score: -0.0851
Test RMSE: 2.6135

Top 5 Chemical Drivers for Drug Potency:
Bit_904    0.015738
Bit_447    0.012990
Bit_362    0.012875
Bit_711    0.012274
Bit_119    0.011021
dtype: float64


### This is Random Forest with aggregated chemical data

In [10]:
chemical_aggregated_baseline_model(df, target='AUC', multi=True, single=False)
chemical_aggregated_baseline_model(df, target='LN_IC50', multi=True, single=False)

C:\Users\Juli\AppData\Local\Temp\ipykernel_18140\2433587977.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()


Starting Cross-Validation on Aggregated Chemical Data...
--- Chemical features baseline performance ---
R² Score: -0.0191
RMSE:     0.1053
------------------------------------

Top 5 chemical drivers for general drug potency:
Bit_1      0.027660
Bit_726    0.024793
Bit_389    0.023588
Bit_136    0.017737
Bit_64     0.017296
dtype: float64


C:\Users\Juli\AppData\Local\Temp\ipykernel_18140\2433587977.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()


Starting Cross-Validation on Aggregated Chemical Data...
--- Chemical features baseline performance ---
R² Score: 0.0384
RMSE:     2.3186
------------------------------------

Top 5 chemical drivers for general drug potency:
Bit_64     0.022385
Bit_849    0.020167
Bit_887    0.019584
Bit_33     0.019394
Bit_1      0.019341
dtype: float64
